# Embeddings

Notebook 1 decided *what* gets embedded — a `small_to_big` chunk: a narrow
sentence window carries the vector, the full abstract carries the meaning
returned to a user. This notebook turns those chunks into actual numbers, and
opens up what a vector is before treating it as a black box.


**Batching.** `text-embedding-3-small` takes a list of strings per request, so
100 chunks is one API call, not 100. `readnext.embed.embed_texts` batches at
`EMBEDDING_BATCH_SIZE`, and every text is looked up in an on-disk cache first
— keyed on a hash of `(model, text)` — so nothing gets embedded twice.


In [1]:
from readnext.config import PAPERS_FILE
from readnext.corpus import chunk, load
from readnext.embed import embed_texts

papers = load(PAPERS_FILE)
chunks = chunk(papers, strategy="small_to_big")
len(papers), len(chunks)


(1693, 12320)

**First embed.** This is the one call in the notebook that actually costs
money — everything after it should be free. Time it, so the cache's effect on
a re-run is visible rather than assumed.


In [2]:
import time

t0 = time.time()
vectors = embed_texts([c.text for c in chunks])
elapsed = time.time() - t0
vectors.shape, f"{elapsed:.1f}s"


((12320, 1536), '59.5s')

**The cache pays off on re-run.** Same chunks, same model — every lookup
should hit the cache and no request should reach the API. Re-running this
notebook, or this cell, costs nothing.


In [3]:
t0 = time.time()
vectors_again = embed_texts([c.text for c in chunks])
elapsed = time.time() - t0

import numpy as np
assert np.array_equal(vectors, vectors_again)
f"{elapsed:.3f}s for {len(chunks)} chunks, all cached"


'0.841s for 12320 chunks, all cached'

**What did this cost.** `embed_texts` appends one line to `COST_LOG_FILE` per
run that actually reaches the API — the re-run above added nothing. `cache_size`
counts vectors sitting on disk right now.


In [4]:
from readnext.embed import cache_size, total_cost

cache_size(), f"${total_cost():.4f}"


(12320, '$0.0163')

**Cosine similarity, by hand.** A vector store will do this at scale later,
but the operation itself is nothing more than a normalized dot product. Writing
it out once with plain numpy — no library call standing between you and the
number — is what makes the rest of the project stop feeling like magic.


In [5]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between a single vector `a` and each row of `b`."""
    a_norm = a / np.linalg.norm(a)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return b_norm @ a_norm


**One vector per paper.** Each paper contributed one or more chunk vectors
(one per sentence window); average them to get a single paper-level vector for
the nearest-neighbour demo below. `search.py` in notebook 3 will index at the
chunk level — this is just for a human to eyeball.


In [6]:
from collections import defaultdict

chunk_rows_by_paper = defaultdict(list)
for i, c in enumerate(chunks):
    chunk_rows_by_paper[c.paper_id].append(i)

paper_ids = [p.id for p in papers]
paper_vectors = np.stack([
    vectors[chunk_rows_by_paper[pid]].mean(axis=0) for pid in paper_ids
])
paper_vectors.shape


(1693, 1536)

**Nearest neighbours.** Pick one paper, rank every other paper by cosine
similarity to it, and read the top few titles. If the strategy and the model
are doing their job, the neighbours should be recognizably about the same
thing — no label ever told the model that.


In [7]:
def nearest_papers(paper_id: str, k: int = 5):
    idx = paper_ids.index(paper_id)
    sims = cosine_similarity(paper_vectors[idx], paper_vectors)
    order = np.argsort(-sims)
    order = [i for i in order if paper_ids[i] != paper_id][:k]
    return [(papers[i].title, round(float(sims[i]), 3)) for i in order]

query_paper = papers[0]
print(query_paper.title)
print("-" * 60)
for title, score in nearest_papers(query_paper.id):
    print(f"{score:.3f}  {title}")


How to Train a Critic Stably and Efficiently
------------------------------------------------------------
0.708  Hints, Critics, and Teachers: Prior Injection for Sparse-Reward RL in Vision-Language Math Reasoning
0.694  Beyond Teacher Likelihood: Group-Calibrated On-Policy Distillation for Long-Context Reasoning
0.689  Let Credit Follow Computation: Architecture-Aware Credit Transport for Large Language Model Reinforcement Learning
0.683  Learning When to Think: Adaptive Reasoning for Test-Time Compute Allocation
0.682  Beyond Imitation: Self-Improving Robot Policies via Off-Policy Q-Planning


**Failure mode: negation.** Embeddings encode topic, not logic. A query for
"not about transformers" should — if the model understood negation — land far
from papers about transformers. It doesn't: the embedding is dominated by the
word "transformers" regardless of what's in front of it. Worth seeing once,
because it explains a whole class of retrieval bugs before you hit one.


In [8]:
transformer_papers = [p for p in papers if "transformer" in p.title.lower()]
other_papers = [p for p in papers if "transformer" not in p.title.lower()][:200]
sample = transformer_papers[:5] + other_papers

sample_vectors = embed_texts([p.abstract for p in sample])
negation_vector = embed_texts(["not about transformers"])[0]

sims = cosine_similarity(negation_vector, sample_vectors)
order = np.argsort(-sims)[:5]
for i in order:
    tag = "[transformer paper]" if sample[i] in transformer_papers else "[other]"
    print(f"{sims[i]:.3f}  {tag}  {sample[i].title}")


0.344  [transformer paper]  Align, Unify, Suppress, Route: A Coherentist View of Transformer Computation
0.326  [transformer paper]  The Communication Map of a Transformer
0.324  [transformer paper]  Sparse Token Routing in Efficient Transformers
0.272  [other]  TANGO: Token-Aggregated Nonlinear Gating Operators for Natural and Formal Language Modeling
0.258  [transformer paper]  ChebBooster: A Training-Free Approach for Efficient Diffusion Transformer Inference via Chebyshev-Inspired Extrapolation


If the top hits are tagged `[transformer paper]`, that confirms it: the
embedding for "not about transformers" is nearest to exactly the papers it
asked to exclude. Negation is a job for the query-understanding step in
notebook 6 (an explicit `exclude_terms` filter), not for the embedding to
solve on its own.

**Done.** The corpus is embedded, cached, and costed. `embed_texts` re-runs
for free, and cosine similarity — computed by hand — recovers sensible nearest
neighbours and exposes the negation failure mode the LLM will need to work
around later.
